# Kepler's Equation and Convergence Analysis
# Análisis de Convergencia y la Ecuación de Kepler

This notebook covers the numerical analysis of Kepler's transcendental equation:
$$M = E - e \sin E$$

We will study the geometric relationship between anomalies, analyze the convergence of the Newton-Raphson method, and explore orbital velocity using the vis-viva equation.

Este cuaderno abarca el análisis numérico de la ecuación trascendente de Kepler, la relación geométrica entre las anomalías, la convergencia del método Newton-Raphson y la velocidad mediante la ecuación de la vis-viva.

In [1]:
# Initialize environment / Inicializar entorno
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from orbital.i18n import Locale
from orbital.gamification import ProgressTracker
from orbital.kepler import solve_kepler_equation, true_anomaly_from_eccentric
from orbital.constants import MU_EARTH, EARTH_RADIUS

tracker = ProgressTracker()
locale = Locale("en")
print("Environment initialized successfully. Action solve_kepler logged.")

ModuleNotFoundError: No module named 'orbital'

## 1. Geometric Interpretation of Anomalies
## 1. Interpretación Geométrica de Anomalías

The eccentric anomaly $E$ acts as an auxiliary angle mapping a projection on the auxiliary circle of radius $a$ (semi-major axis) to the actual satellite position on the ellipse.

In [ ]:
output_geom = widgets.Output()

slider_e_geom = widgets.FloatSlider(value=0.5, min=0.0, max=0.9, step=0.01, description='e')
slider_E_geom = widgets.FloatSlider(value=60, min=0, max=360, step=1, description='E (deg)')
lang_dropdown = widgets.Dropdown(options=[('English', 'en'), ('Español', 'es'), ('简体中文', 'zh')], value='en', description='Language')

def draw_geometry(*args):
    e = slider_e_geom.value
    E_deg = slider_E_geom.value
    lang = lang_dropdown.value
    
    loc = Locale(lang)
    E = np.radians(E_deg)
    a = 1.0
    b = a * np.sqrt(1 - e**2)
    c = a * e
    
    theta = np.linspace(0, 2*np.pi, 500)
    x_elipse = a * np.cos(theta) - c
    y_elipse = b * np.sin(theta)
    x_circ = a * np.cos(theta)
    y_circ = a * np.sin(theta)
    
    x_sat = a * np.cos(E) - c
    y_sat = b * np.sin(E)
    x_aux = a * np.cos(E)
    y_aux = a * np.sin(E)
    
    M = E - e * np.sin(E)
    nu = true_anomaly_from_eccentric(E, e)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x_elipse, y=y_elipse, mode='lines', line=dict(color='#00ff88', width=2.5), name='Ellipse'))
    fig.add_trace(go.Scatter(x=x_circ, y=y_circ, mode='lines', line=dict(color='#555', width=1, dash='dash'), name='Auxiliary Circle'))
    fig.add_trace(go.Scatter(x=[-c], y=[0], mode='markers', marker=dict(color='#4488ff', size=15), name=loc.t("ui.earth")))
    fig.add_trace(go.Scatter(x=[x_sat], y=[y_sat], mode='markers', marker=dict(color='#ff1493', size=12, symbol='star'), name=loc.t("ui.satellite")))
    fig.add_trace(go.Scatter(x=[x_aux, x_sat], y=[y_aux, y_sat], mode='lines', line=dict(color='#ffd700', width=1.5, dash='dot'), name='Projection'))
    fig.add_trace(go.Scatter(x=[-c, x_sat], y=[0, y_sat], mode='lines', line=dict(color='#ff6b35', width=2), name='Radius Vector'))
    
    fig.update_layout(
        template='plotly_dark',
        xaxis=dict(scaleanchor='y', scaleratio=1, title='x/a'),
        yaxis=dict(title='y/a'),
        height=450, width=650,
        margin=dict(t=30, b=30, l=30, r=30),
        paper_bgcolor='rgb(10,10,25)',
        font=dict(family='Inter, sans-serif')
    )
    
    with output_geom:
        output_geom.clear_output(wait=True)
        fig.show()
        print(f"Results:")
        print(f"  Eccentric Anomaly E = {E_deg:.1f} deg")
        print(f"  Mean Anomaly      M = {np.degrees(M):.1f} deg")
        print(f"  True Anomaly      nu = {np.degrees(nu):.1f} deg")
        
    if e > 0.1:
        tracker.record_action("solve_kepler")

slider_e_geom.observe(draw_geometry, 'value')
slider_E_geom.observe(draw_geometry, 'value')
lang_dropdown.observe(draw_geometry, 'value')

controls = widgets.VBox([lang_dropdown, slider_e_geom, slider_E_geom])
display(controls, output_geom)
draw_geometry()

## 2. Convergence Analysis of Newton-Raphson
## 2. Análisis de Convergencia de Newton-Raphson

Newton-Raphson has quadratic convergence. For extremely high eccentricities (e.g., $e = 0.99$), convergence slows down slightly but remains very fast.

In [ ]:
M_test = np.pi / 3  # 60 degrees
eccentricities_test = [0.1, 0.3, 0.5, 0.7, 0.9, 0.99]
colors_conv = ['#00ff88', '#00bfff', '#ffd700', '#ff6b35', '#ff1493', '#ff0040']

fig_conv = go.Figure()

for e, color in zip(eccentricities_test, colors_conv):
    E = M_test
    errors = []
    
    for it in range(20):
        f = E - e * np.sin(E) - M_test
        errors.append(abs(f))
        if abs(f) < 1e-16:
            break
        f_prime = 1.0 - e * np.cos(E)
        E = E - f / f_prime
        
    fig_conv.add_trace(go.Scatter(
        x=list(range(1, len(errors) + 1)),
        y=errors,
        mode='lines+markers',
        name=f'e = {e}',
        line=dict(color=color, width=2.5),
        marker=dict(size=6)
    ))

fig_conv.update_layout(
    template='plotly_dark',
    xaxis_title='Iteration',
    yaxis_title='Residual Error |f(E)|',
    yaxis_type='log',
    height=450, width=750,
    paper_bgcolor='rgb(10,10,25)',
    font=dict(family='Inter, sans-serif')
)
fig_conv.add_hline(y=2.2e-16, line_dash='dot', line_color='white')
fig_conv.show()

## 3. Vis-Viva Equation: Velocity vs Altitude
## 3. Ecuación Vis-Viva: Velocidad contra Altitud

The velocity of a satellite in orbit is given by:
$$v = \sqrt{\mu \left(\frac{2}{r} - \frac{1}{a}\right)}$$

In [ ]:
semi_axes = [6778, 10000, 20000, 26571, 42164]
labels = ['LEO (ISS)', 'MEO Low', 'MEO High', 'GPS', 'GEO']
colors_vv = ['#00ff88', '#00bfff', '#ffd700', '#ff6b35', '#ff1493']

fig_vv = go.Figure()
r_vals = np.linspace(6500, 50000, 1000)

for a, label, color in zip(semi_axes, labels, colors_vv):
    v_sq = MU_EARTH * (2.0 / r_vals - 1.0 / a)
    mask = v_sq > 0
    v = np.sqrt(v_sq[mask])
    
    fig_vv.add_trace(go.Scatter(
        x=(r_vals[mask] - EARTH_RADIUS),
        y=v,
        mode='lines',
        name=f'{label} (a={a} km)',
        line=dict(color=color, width=2.5)
    ))

v_circ = np.sqrt(MU_EARTH / r_vals)
fig_vv.add_trace(go.Scatter(x=(r_vals - EARTH_RADIUS), y=v_circ, mode='lines', name='Circular Velocity', line=dict(color='white', width=2, dash='dash')))

fig_vv.update_layout(
    template='plotly_dark',
    xaxis_title='Altitude [km]',
    yaxis_title='Velocity [km/s]',
    height=480, width=800,
    paper_bgcolor='rgb(10,10,25)',
    font=dict(family='Inter, sans-serif')
)
fig_vv.show()